# Aula 4: encadear operações

Até aqui, cada operação foi uma linha separada, guardando o resultado numa
variável nova. Funciona, e fica ilegível rápido.

Nesta aula toda análise vira uma **sequência de operações encadeadas**, escrita
de cima para baixo dentro de um par de parênteses. Cada linha faz uma coisa, e
você lê o que aconteceu na ordem em que aconteceu.

O roteiro:

1. ver o problema que o encadeamento resolve;
2. aprender os verbos, um a um: filtrar, criar coluna, ordenar, escolher
   colunas, agrupar e agregar;
3. montar um pipeline inteiro para responder a uma pergunta.


In [ ]:
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

URL = "https://raw.githubusercontent.com/jtrecenti/202662-cdad2/main/dados"


## A base

Recursos criminais do TJSP, com o regime inicial e a pena lidos da ementa.

```python
import juscraper as jus

tjsp = jus.scraper("tjsp")
acordaos = tjsp.cjsg('"apelacao criminal" E "regime inicial"', paginas=range(1, 26))
```


In [ ]:
criminal = pd.read_csv(f"{URL}/tjsp_cjsg_criminal.csv")
criminal.head(3)


In [ ]:
criminal.info()


> `regime_inicial` e `pena_anos` foram lidos do texto da ementa, e nenhum dos
> dois vem completo: o regime aparece em cerca de 70% dos acórdãos e a pena em
> 45%. `pena_anos` ainda traz valores implausíveis, porque a leitura pega o
> primeiro número seguido de "anos" que encontra. Vamos lidar com isso.


## A pergunta de pesquisa

> Nas apelações criminais do TJSP, a proporção de acórdãos que mencionam
> reincidência varia conforme o regime inicial fixado?

## O problema: uma variável para cada passo

Resolvendo do jeito que fizemos até agora, com uma variável nova por operação:


In [ ]:
apelacoes = criminal[criminal["classe"] == "Apelação Criminal"]
com_regime = apelacoes.dropna(subset=["regime_inicial"])
fechado = com_regime[com_regime["regime_inicial"] == "fechado"]
semiaberto = com_regime[com_regime["regime_inicial"] == "semiaberto"]
aberto = com_regime[com_regime["regime_inicial"] == "aberto"]

pd.Series({
    "fechado": fechado["houve_reincidencia"].mean(),
    "semiaberto": semiaberto["houve_reincidencia"].mean(),
    "aberto": aberto["houve_reincidencia"].mean(),
}).round(3)


Funciona, e tem três problemas:

1. **seis variáveis** que existem só para chegar num resultado, e que continuam
   ocupando memória e atrapalhando a leitura do resto do notebook;
2. **nomes intermediários** como `com_regime` que não querem dizer nada e que
   você vai reaproveitar por engano daqui a três células;
3. **não escala**: se aparecesse um quarto regime, seria preciso escrever mais
   uma linha e lembrar de incluí-la no resultado.


## A mesma coisa, encadeada

Agora o mesmo resultado, escrito como uma sequência:


In [ ]:
(
    criminal
    .query("classe == 'Apelação Criminal'")
    .dropna(subset=["regime_inicial"])
    .groupby("regime_inicial")
    .agg(proporcao=("houve_reincidencia", "mean"))
    .round(3)
)


Leia de cima para baixo: pegue `criminal`, fique só com as apelações, descarte
quem não tem regime, junte por regime, e calcule a proporção. Nenhuma variável
intermediária, e a ordem das operações é a ordem das linhas.

### Por que os parênteses

Em Python, dentro de um par de parênteses você pode quebrar a linha à vontade.
Sem eles, `criminal` seguido de uma quebra de linha e `.query(...)` é erro de
sintaxe. Os parênteses existem só para deixar você pôr uma operação por linha.

O formato que vamos usar sempre é este:

```python
resultado = (
    tabela
    .operacao_1(...)
    .operacao_2(...)
)
```

Abre parêntese, o nome da tabela sozinho na primeira linha, e daí em diante uma
operação por linha, cada uma começando com ponto.


## Os verbos

São seis operações, e quase toda análise descritiva é uma combinação delas.

### 1. `.query()`: escolher linhas

Recebe a condição escrita como texto. Dentro das aspas, os nomes das colunas
aparecem sem `df[...]`, e o texto que você compara vai entre aspas simples.


In [ ]:
criminal.query("regime_inicial == 'fechado'").shape


Para combinar condições, use `and`, `or` e `not`, por extenso:


In [ ]:
criminal.query("regime_inicial == 'fechado' and houve_reincidencia").shape


**Agora você.** Fique só com os acórdãos de tráfico em que houve confissão.


In [ ]:
criminal.query("eh_trafico ________ houve_confissao").shape


Para usar uma variável do Python dentro da condição, ponha `@` na frente dela:


In [ ]:
regime_alvo = "semiaberto"

criminal.query("regime_inicial == @regime_alvo").shape


### 2. `.assign()`: criar colunas

`.assign(nome_da_coluna=...)` devolve uma cópia da tabela com a coluna nova. Ele
não altera a tabela original, e é por isso que serve para encadear.


In [ ]:
(
    criminal
    .assign(ementa_longa=criminal["n_palavras_ementa"] > 200)
    [["processo", "n_palavras_ementa", "ementa_longa"]]
    .head(3)
)


Só que escrever `criminal[...]` lá dentro estraga o encadeamento: se antes do
`.assign` houve um filtro, `criminal` ainda é a tabela inteira, e as duas não
têm mais o mesmo número de linhas.

A solução é `lambda d:`, que quer dizer "a tabela como ela está **neste ponto**
da sequência". O `d` é só um nome, e podia ser qualquer outro.


In [ ]:
(
    criminal
    .query("regime_inicial == 'fechado'")
    .assign(ementa_longa=lambda d: d["n_palavras_ementa"] > 200)
    [["processo", "regime_inicial", "n_palavras_ementa", "ementa_longa"]]
    .head(3)
)


**Agora você.** Crie a coluna `pena_alta`, verdadeira quando `pena_anos` for maior que 8, usando `lambda`.


In [ ]:
(
    criminal
    .assign(pena_alta=lambda d: d["________"] > 8)
    [["processo", "pena_anos", "pena_alta"]]
    .head(3)
)


Dá para criar várias colunas de uma vez, separando por vírgula. E uma coluna
criada num `.assign` pode ser usada na seguinte, desde que seja com `lambda`:


In [ ]:
(
    criminal
    .assign(
        pena_meses=lambda d: d["pena_anos"] * 12,
        pena_meses_arredondada=lambda d: d["pena_meses"].round(0),
    )
    [["processo", "pena_anos", "pena_meses", "pena_meses_arredondada"]]
    .head(3)
)


### 3. `.sort_values()`: ordenar

`by=` diz por qual coluna, e `ascending=False` inverte para o maior primeiro.


In [ ]:
(
    criminal
    .sort_values("n_palavras_ementa", ascending=False)
    [["processo", "comarca", "n_palavras_ementa"]]
    .head(5)
)


**Agora você.** Ordene pela pena, da maior para a menor, e olhe as cinco primeiras. Repare no que aparece: a leitura automática da pena erra em alguns acórdãos.


In [ ]:
(
    criminal
    .sort_values("________", ascending=________)
    [["processo", "pena_anos", "regime_inicial"]]
    .head(5)
)


### 4. Escolher colunas

Duas chaves com uma lista de nomes dentro devolvem só aquelas colunas, na ordem
que você pediu. Já apareceu nos exemplos acima:


In [ ]:
(
    criminal
    [["processo", "comarca", "regime_inicial", "pena_anos"]]
    .head(3)
)


### 5. `.groupby()` e `.agg()`: agregar por grupo

Esta é a operação nova de verdade. `.groupby("coluna")` separa a tabela em
pedaços, um por valor da coluna, e `.agg(...)` calcula uma estatística em cada
pedaço, devolvendo uma linha por grupo.

A forma de escrever é `nome_da_saida=("coluna_de_entrada", "estatistica")`:


In [ ]:
(
    criminal
    .groupby("regime_inicial")
    .agg(
        n=("processo", "size"),
        mediana_palavras=("n_palavras_ementa", "median"),
    )
)


`"size"` conta as linhas do grupo. As outras estatísticas são as mesmas da aula
3, escritas como texto: `"mean"`, `"median"`, `"std"`, `"min"`, `"max"`,
`"sum"`, `"nunique"`.

E vale lembrar da aula 3: a média de uma coluna de verdadeiro e falso é a
proporção. Isso funciona igual dentro do `.agg`.


**Agora você.** Acrescente ao resumo acima a proporção de acórdãos com reincidência e a proporção de tráfico.


In [ ]:
(
    criminal
    .groupby("regime_inicial")
    .agg(
        n=("processo", "size"),
        prop_reincidencia=("houve_reincidencia", "________"),
        prop_trafico=("________", "mean"),
    )
    .round(3)
)


Dá para agrupar por mais de uma coluna, passando uma lista. O resultado ganha
uma linha por combinação:


In [ ]:
(
    criminal
    .groupby(["regime_inicial", "houve_confissao"])
    .agg(n=("processo", "size"))
    .head(6)
)


### 6. `.reset_index()`: voltar a ser uma tabela comum

Depois de um `groupby`, a coluna de agrupamento vira o **índice** do resultado,
e não uma coluna normal. Repare que `regime_inicial` está fora da tabela, à
esquerda, em negrito. Isso atrapalha se você quiser continuar encadeando.
`.reset_index()` traz o índice de volta para dentro:


In [ ]:
(
    criminal
    .groupby("regime_inicial")
    .agg(n=("processo", "size"))
    .reset_index()
)


Com o índice de volta, dá para filtrar e ordenar o resultado como qualquer outra
tabela, o que é justamente o que vamos fazer no pipeline completo.


## Montando o pipeline

Voltando à pergunta: a proporção de menção a reincidência varia conforme o
regime inicial?

Duas coisas ainda faltam. Primeiro, o regime é **ordinal**, e queremos a tabela
na ordem aberto, semiaberto, fechado, e não em ordem alfabética. Isso é a
categórica ordenada da aula 2, criada aqui dentro do `.assign`. Segundo,
`groupby` sobre categórica traz todas as categorias declaradas, e `observed=True`
mantém só as que aparecem.


In [ ]:
resumo = (
    criminal
    .query("classe == 'Apelação Criminal'")
    .dropna(subset=["regime_inicial"])
    .assign(
        regime=lambda d: pd.Categorical(
            d["regime_inicial"],
            categories=["aberto", "semiaberto", "fechado"],
            ordered=True,
        )
    )
    .groupby("regime", observed=True)
    .agg(
        n=("processo", "size"),
        prop_reincidencia=("houve_reincidencia", "mean"),
        prop_confissao=("houve_confissao", "mean"),
        prop_trafico=("eh_trafico", "mean"),
    )
    .round(3)
)

resumo


A leitura é direta: a menção a reincidência sobe conforme o regime fica mais
severo. Isso não é surpresa, é quase a definição legal do regime, e serve para
conferir que a leitura das variáveis está coerente.

E o que **não** dá para concluir: nada sobre causalidade, e nada sobre acórdãos
em que o regime não foi identificado, que são cerca de 30% da base.


**Agora você.** Monte um resumo parecido, agora por `camara`, mantendo só as câmaras com pelo menos 15 acórdãos e ordenando da maior proporção de reincidência para a menor. Você vai precisar de `.reset_index()`, `.query()` e `.sort_values()` depois do `.agg()`.


In [ ]:
(
    criminal
    .query("classe == 'Apelação Criminal'")
    .dropna(subset=["camara"])
    .groupby("________")
    .agg(
        n=("processo", "size"),
        prop_reincidencia=("houve_reincidencia", "________"),
    )
    .________()
    .query("n >= ________")
    .sort_values("________", ascending=False)
    .round(3)
)


## Quatro erros que você vai cometer

### 1. Esquecer o parêntese de abertura

Sem os parênteses, a quebra de linha encerra o comando:


In [ ]:
codigo = '''
criminal
.query("eh_trafico")
'''

try:
    exec(codigo)
except SyntaxError as erro:
    print("SyntaxError:", erro)


### 2. Usar a tabela original dentro do encadeamento

Uma coluna criada no meio da sequência não existe na tabela original. Referir a
ela pelo nome da tabela levanta erro:


In [ ]:
try:
    (
        criminal
        .assign(pena_meses=lambda d: d["pena_anos"] * 12)
        .assign(pena_alta=criminal["pena_meses"] > 96)
    )
except KeyError as erro:
    print("KeyError:", erro)


Este erro é o caso fácil, porque ele aparece. O caso difícil é quando **não**
aparece. Abaixo, as duas versões rodam sem reclamar, e dão resultados
diferentes: a da esquerda usa `criminal`, e o pandas casa as linhas pelo número
do índice, que depois do `reset_index` já não é o mesmo.


In [ ]:
errado = (
    criminal
    .query("regime_inicial == 'fechado'")
    .reset_index(drop=True)
    .assign(longa=criminal["n_palavras_ementa"] > 200)
)

certo = (
    criminal
    .query("regime_inicial == 'fechado'")
    .reset_index(drop=True)
    .assign(longa=lambda d: d["n_palavras_ementa"] > 200)
)

pd.Series({
    "linhas": len(certo),
    "em que as duas versões concordam": (errado["longa"] == certo["longa"]).mean(),
}).round(3)


Quase 30% das linhas ficaram com o valor de outro acórdão, sem aviso nenhum.
Por isso a regra é sem exceção: dentro de um encadeamento, olhe para as colunas
com `lambda d:`, nunca pelo nome da tabela.


### 3. Achar que `.assign` altera a tabela

`.assign` devolve uma cópia. Se você não guardar o resultado, a coluna não
existe fora do encadeamento:


In [ ]:
criminal.assign(teste=1)

"teste" in criminal.columns


### 4. Encadear demais

Sequência com quinze operações é tão ruim de ler quanto quinze variáveis soltas.
Quando o encadeamento passar de umas oito linhas, ou quando um resultado
intermediário for usado em dois lugares, quebre em duas partes com um nome que
signifique alguma coisa, como fizemos com `resumo`.


## Exercícios

### Exercício 1

A pena lida da ementa tem valores implausíveis, como penas acima de 40 anos, que
vêm de a leitura pegar um número errado. Monte um encadeamento que descarte as
penas ausentes e as maiores que 30 anos, e devolva mediana, média e desvio
padrão da pena por regime inicial.


In [ ]:
(
    criminal
    .dropna(subset=["pena_anos", "regime_inicial"])
    .query("pena_anos ________ 30")
    .groupby("________")
    .agg(
        n=("processo", "size"),
        mediana=("pena_anos", "________"),
        media=("pena_anos", "mean"),
        desvio=("pena_anos", "________"),
    )
    .round(2)
)


### Exercício 2

Quais são as cinco comarcas com mais apelações criminais nesta base, e qual a
proporção de tráfico em cada uma?


In [ ]:
(
    criminal
    .query("classe == 'Apelação Criminal'")
    .groupby("________")
    .agg(n=("processo", "size"), prop_trafico=("eh_trafico", "________"))
    .reset_index()
    .sort_values("________", ascending=False)
    .head(________)
    .round(3)
)


### Exercício 3

Escreva, em duas ou três linhas, uma pergunta descritiva que **não** dá para
responder com esta base, e diga que variável faltaria. Não precisa programar.


In [ ]:
# exercício 3 (responda em célula de texto)


## O que ficou

| verbo | para quê |
|---|---|
| `.query("...")` | escolher linhas por uma condição |
| `.dropna(subset=[...])` | descartar linhas sem valor numa coluna |
| `.assign(nova=lambda d: ...)` | criar coluna |
| `[["a", "b"]]` | escolher colunas |
| `.sort_values("a", ascending=False)` | ordenar |
| `.groupby("a").agg(saida=("b", "mean"))` | uma linha por grupo |
| `.reset_index()` | tirar o agrupamento do índice |
| `.head(n)` | cortar as primeiras linhas |

Duas regras que valem sempre: dentro do encadeamento, olhe para as colunas com
`lambda d:`, e não pelo nome da tabela original. E quando a sequência ficar
longa demais para caber na tela, quebre em duas.
